# Extraction d'un bulletin scolaire tunisien avec Google Cloud Vision API

Ce notebook :
1. Envoie une image de bulletin scolaire (bulletin tunisien, en arabe) à l'API **Google Cloud Vision** (OCR).
2. Analyse le texte brut renvoyé pour en extraire les champs structurés (identité de l'élève, notes par matière, résultats généraux).
3. Écrit le tout dans des fichiers **CSV**.

**Champs extraits :**
- Informations générales : établissement, année scolaire, nom, date/lieu de naissance, classe, identifiant unique, numéro d'ordre, nombre d'élèves.
- Tableau des notes : matière, moyenne + rang pour le 1er semestre, le 2e semestre et l'année.
- Résultats généraux : moyenne/rang/mentions par semestre, moyenne annuelle, décision du conseil de classe, nom du directeur.

> ⚠️ L'OCR d'un document arabe scanné n'est jamais parfait à 100 %. Le parsing ci-dessous est volontairement tolérant (regex sur les libellés) mais vérifiez toujours le CSV de sortie contre l'image originale.


## 1. Installation des dépendances

Il vous faut le package `google-cloud-vision`.


In [13]:
%pip install --quiet --upgrade google-cloud-vision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Authentification Google Cloud

Deux façons de vous authentifier :

**Option A — Clé de compte de service (recommandé pour un notebook local)**
1. Dans la console Google Cloud, créez un compte de service avec le rôle *Cloud Vision AI Service Agent* (ou *Editor*).
2. Téléchargez la clé JSON du compte de service.
3. Renseignez le chemin vers ce fichier ci-dessous.

**Option B — `gcloud auth application-default login`**
Si vous avez déjà lancé cette commande dans un terminal, laissez `SERVICE_ACCOUNT_JSON_PATH = None` : les identifiants par défaut de l'environnement seront utilisés automatiquement.


In [14]:
import os

# --- À adapter ---
SERVICE_ACCOUNT_JSON_PATH = None  # mettez None pour utiliser les Application Default Credentials
IMAGE_PATH = "bulletin.jpg"                              # chemin de l'image du bulletin à analyser
OUTPUT_INFO_CSV = "bulletin_infos.csv"                    # CSV des informations générales (1 ligne)
OUTPUT_GRADES_CSV = "bulletin_notes.csv"                  # CSV du tableau des notes (1 ligne par matière)

if SERVICE_ACCOUNT_JSON_PATH:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SERVICE_ACCOUNT_JSON_PATH
    print(f"Utilisation de la clé de service : {SERVICE_ACCOUNT_JSON_PATH}")
else:
    print("Utilisation des Application Default Credentials (gcloud auth application-default login)")


Utilisation des Application Default Credentials (gcloud auth application-default login)


In [15]:
import google.cloud
print("Google Cloud library is installed.")

Google Cloud library is installed.


## 3. Appel à l'API Google Cloud Vision (OCR)

On utilise `document_text_detection`, plus adapté qu'un simple `text_detection` pour les documents structurés (meilleure détection des blocs/paragraphes et de l'arabe).


In [16]:
from google.cloud import vision

def extract_text_from_image(image_path: str) -> str:
    client = vision.ImageAnnotatorClient()

    with open(image_path, "rb") as f:
        content = f.read()

    image = vision.Image(content=content)

    # image_context avec language_hints améliore la reconnaissance de l'arabe
    image_context = vision.ImageContext(language_hints=["ar"])

    response = client.document_text_detection(image=image, image_context=image_context)

    if response.error.message:
        raise RuntimeError(
            f"Erreur Google Vision API : {response.error.message}"
        )

    return response.full_text_annotation.text


raw_text = extract_text_from_image(IMAGE_PATH)
print(raw_text)


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

## 4. Analyse (parsing) du texte extrait

Le texte OCR est découpé en lignes et on cherche, pour chaque champ attendu, le libellé arabe correspondant suivi de sa valeur.
La liste `SUBJECTS` couvre les matières habituelles d'un bulletin tunisien de lycée ; adaptez-la si votre bulletin contient d'autres matières.


In [ ]:
import re

# Matières attendues dans le tableau des notes (à adapter si nécessaire)
SUBJECTS = [
    "عربية",
    "فرنسية",
    "أنقليزية",
    "تاريخ",
    "جغرافيا",
    "تفكير إسلامي",
    "تربية مدنية",
    "رياضيات",
    "علوم فيزيائية",
    "علوم الحياة والأرض",
    "تكنولوجيا",
    "تربية بدنية",
]


def _grab_line(label: str, text: str):
    pattern = re.escape(label) + r"\s*:?\s*([^\n]+)"
    m = re.search(pattern, text)
    return m.group(1).strip() if m else None


def parse_general_info(text: str) -> dict:
    info = {
        "etablissement": _grab_line("الجمهورية التونسية", text),
        "annee_scolaire": _grab_line("السنة الدراسية", text),
        "nom_prenom": _grab_line("الاسم واللقب", text),
        "date_naissance": _grab_line("تاريخ الولادة", text),
        "lieu_naissance": _grab_line("مكانها", text),
        "classe": _grab_line("القسم", text),
        "identifiant_unique": _grab_line("المعرف الوحيد", text),
        "numero_ordre": _grab_line("العدد الرتبي", text),
        "nombre_eleves": _grab_line("التلاميذ", text),
    }

    # Résultats généraux : isolés à partir de "النتائج العامة" pour éviter
    # toute confusion avec l'en-tête du tableau des notes ("السداسي الأول ...").
    idx = text.find("النتائج العامة")
    results_section = text[idx:] if idx != -1 else text

    info["semestre1_resultat"] = _grab_line("السداسي الأول", results_section)
    info["semestre2_resultat"] = _grab_line("السداسي الثاني", results_section)
    info["moyenne_annuelle_resultat"] = _grab_line("المعدل السنوي", results_section)
    info["decision_conseil_classe"] = _grab_line(
        "قرار مجلس القسم وملاحظات المدير", results_section
    )
    info["directeur"] = _grab_line("المدير(ة)", results_section) or _grab_line(
        "المدير", results_section
    )
    return info


def parse_grades_table(text: str) -> list:
    rows = []
    for subject in SUBJECTS:
        pattern = (
            re.escape(subject)
            + r"\s+([\d.]+)\s*-\s*(\d+)"   # semestre 1 : moyenne - rang
            + r"\s+([\d.]+)\s*-\s*(\d+)"   # semestre 2 : moyenne - rang
            + r"\s+([\d.]+)\s*-\s*(\d+)"   # annuel     : moyenne - rang
        )
        m = re.search(pattern, text)
        if m:
            rows.append({
                "matiere": subject,
                "semestre1_moyenne": m.group(1),
                "semestre1_rang": m.group(2),
                "semestre2_moyenne": m.group(3),
                "semestre2_rang": m.group(4),
                "annuel_moyenne": m.group(5),
                "annuel_rang": m.group(6),
            })
        else:
            # La matière n'a pas été trouvée / OCR imparfait : on garde une ligne
            # vide pour que l'utilisateur puisse la corriger manuellement dans le CSV.
            rows.append({
                "matiere": subject,
                "semestre1_moyenne": "",
                "semestre1_rang": "",
                "semestre2_moyenne": "",
                "semestre2_rang": "",
                "annuel_moyenne": "",
                "annuel_rang": "",
            })
    return rows


general_info = parse_general_info(raw_text)
grades = parse_grades_table(raw_text)

print("Informations générales :")
for k, v in general_info.items():
    print(f"  {k}: {v}")

print("\nNotes par matière :")
for row in grades:
    print(f"  {row}")


## 5. Écriture des résultats en CSV

Deux fichiers sont produits :
- `bulletin_infos.csv` : une seule ligne avec toutes les informations générales de l'élève.
- `bulletin_notes.csv` : une ligne par matière avec les moyennes et rangs.

`encoding="utf-8-sig"` est utilisé pour que l'arabe s'affiche correctement si le CSV est ensuite ouvert dans Excel.


In [ ]:
import csv

# --- Informations générales ---
with open(OUTPUT_INFO_CSV, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(general_info.keys()))
    writer.writeheader()
    writer.writerow(general_info)

# --- Tableau des notes ---
with open(OUTPUT_GRADES_CSV, "w", newline="", encoding="utf-8-sig") as f:
    fieldnames = [
        "matiere",
        "semestre1_moyenne", "semestre1_rang",
        "semestre2_moyenne", "semestre2_rang",
        "annuel_moyenne", "annuel_rang",
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(grades)

print(f"✅ Écrit : {OUTPUT_INFO_CSV}")
print(f"✅ Écrit : {OUTPUT_GRADES_CSV}")


## 6. (Optionnel) Vérification rapide avec pandas


In [ ]:
import pandas as pd

display(pd.read_csv(OUTPUT_INFO_CSV))
display(pd.read_csv(OUTPUT_GRADES_CSV))


## Notes & limites

- Si certaines matières ou certains champs restent vides dans le CSV, c'est probablement que l'OCR n'a pas reconnu le libellé exactement (ligature, saut de ligne inattendu, qualité d'image). Dans ce cas :
  - Essayez une image mieux cadrée / de meilleure résolution.
  - Affichez `raw_text` (cellule 3) et ajustez les libellés dans `SUBJECTS` ou les regex de `parse_general_info` / `parse_grades_table` en conséquence.
- Pour traiter plusieurs bulletins d'un coup, il suffit de boucler sur une liste de chemins d'images et d'accumuler les lignes avant l'écriture CSV (une ligne par élève dans `bulletin_infos.csv`, et un CSV `bulletin_notes.csv` avec une colonne supplémentaire `identifiant_unique` pour relier chaque note à l'élève correspondant).
